# 01 - Load and Chunk the Help-Desk FAQ

The goal of this notebook is to load the ABC Grocery help-desk FAQ and split it into chunks, one for each question and answer pair.

I check the chunks here before creating any embeddings. If the chunking is wrong, retrieval will be wrong too, and embedding costs money.

This notebook makes no API calls, so no API key is needed.

## Setup

`autoreload` picks up changes to `src/config.py` without restarting the kernel. All paths and settings come from `config`, so every notebook uses the same values.

In [1]:
# reload src/config.py automatically whenever it changes
%load_ext autoreload
%autoreload 2

# import project settings
import sys
sys.path.append("..")
from src import config

# import packages
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from pathlib import Path



## Load the FAQ

`TextLoader` reads the file into a LangChain `Document`, which holds the text (`page_content`) and information about where it came from (`metadata`). The whole file loads as one document, so I print only the first 500 characters to check that it looks right.

In [2]:
docs = TextLoader(config.DATA_FILE, encoding="utf-8").load()
text = docs[0].page_content
print("Documents loaded:", len(docs))
print("Characters:", len(text))
print("Source file:", Path(docs[0].metadata["source"]).name)
print()
print(text[:500])

Documents loaded: 1
Characters: 9242
Source file: abc-grocery-help-desk-data.md

### 0001

Q: What is ABC Grocery?

A: ABC Grocery is a nationwide supermarket chain that sells fresh groceries and non-food items such as clothes and shoes. We offer in-store shopping, free pickup, and home delivery. Non-members pay $10 per delivery; Delivery Club members get unlimited free deliveries for a flat annual fee.



### 0002

Q: How do I find the nearest ABC Grocery store?

A: Use the Store Locator in the app or on our website. Enter your ZIP/postcode to see nearby stores, addresses, 


## Split into chunks

Each Q&A pair starts with a `###` header such as `### 0001`, so I split on that header to get one chunk per question.

With `strip_headers=True`, the header is moved out of the text and into the metadata as `id`. The id isn't embedded, because a number like "0001" has no meaning to the embedding model, but it stays attached to the chunk. Later this lets me check whether the retriever returned the right FAQ.

In [3]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("###", "id")],
    strip_headers=True,
)

chunks = splitter.split_text(text)

print("Chunks:", len(chunks))
print("Metadata:", chunks[0].metadata)
print()
print(chunks[0].page_content)

Chunks: 32
Metadata: {'id': '0001'}

Q: What is ABC Grocery?  
A: ABC Grocery is a nationwide supermarket chain that sells fresh groceries and non-food items such as clothes and shoes. We offer in-store shopping, free pickup, and home delivery. Non-members pay $10 per delivery; Delivery Club members get unlimited free deliveries for a flat annual fee.


## Check the chunks

Before embedding, I check that the split worked:

- **Ids:** there should be 32 unique ids, from 0001 to 0032. A set removes duplicates, so if the set has as many ids as there are chunks, no id appears twice. Comparing it with the expected set also catches gaps or typos.
- **Empty chunks:** a chunk with no text would still be embedded and could be retrieved.
- **Q and A:** every chunk should hold both the question and the answer. If the split happened in the wrong place, they could end up in different chunks.

In [4]:
ids = set()
for chunk in chunks:
    ids.add(chunk.metadata['id'])

expected_ids = set()
for i in range(1,33):
    expected_ids.add(f"{i:04}")

print("Chunks:", len(chunks))
print("Unique ids:",len(ids))
print("Ids match 001 - 0032:", ids == expected_ids)

Chunks: 32
Unique ids: 32
Ids match 001 - 0032: True


In [5]:
empty_chunks = []
missing_q_or_a = []

for chunk in chunks:
    chunk_id = chunk.metadata["id"]
    content = chunk.page_content

    if content.strip() == "":
        empty_chunks.append(chunk_id)

    if "Q:" not in content or "A:" not in content:
        missing_q_or_a.append(chunk_id)

print("Empty chunks:", empty_chunks)
print("Missing Q or A:", missing_q_or_a)

Empty chunks: []
Missing Q or A: []


## Chunk sizes

A very long chunk could cover several topics and match too many questions, and a very short one may not carry enough meaning to be found. I check the shortest, longest and average length in characters.

In [6]:
shortest = len(chunks[0].page_content)
longest = len(chunks[0].page_content)
shortest_id = chunks[0].metadata['id']
longest_id = chunks[0].metadata['id']
total = 0
for chunk in chunks:
    length = len(chunk.page_content)
    total+=length

    if length < shortest:
        shortest = length
        shortest_id = chunk.metadata["id"]
    if length > longest:
        longest = length
        longest_id = chunk.metadata["id"]

print(f"Shortest: {shortest} characters (id {shortest_id})")
print(f"Longest:  {longest} characters (id {longest_id})")
print(f"Average:  {total / len(chunks):.0f} characters")

Shortest: 162 characters (id 0032)
Longest:  381 characters (id 0010)
Average:  276 characters


## Summary

- The FAQ loads as **1 document of 9,242 characters**.
- Splitting on the `###` headers gives **32 chunks**, one per Q&A pair. The header is moved into the metadata as `id`, so each chunk keeps its FAQ id without the id being embedded.
- **All checks passed:** 32 unique ids from 0001 to 0032, no empty chunks, and every chunk has both a question and an answer.
- **Chunk sizes:** 162 to 381 characters, with an average of 276. The shortest is 0032 (jobs) and the longest is 0010 (substitutions policy). All chunks are short and similar in size, so each one covers a single topic, which suits retrieval.

**Next:** embed the chunks and store them in a Chroma vector database (`src/ingest.py`).